## 1. Environment Setup and Library Imports

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# -------------------------------------------------------------------------
# Path resolution to access root directory and 'src' modules
# -------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent if "__file__" not in globals() else Path(__file__).resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import local custom modules
from src.pkldataset import PKLDataset
from src.helpers import set_seed, get_model, train_model, eval_model

# Hardware configuration and loss function
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

# Random seeds for evaluation (10 seeds matching benchmarks)
seeds = [101, 202, 303, 404, 505, 606, 707, 808, 909, 1001]

# Base sheet thickness used for HC training subsets
source_thickness = "T197"

# -------------------------------------------------------------------------
# Dataset Paths & Configurations (Late Fusion: HC T197 -> HC Targets)
# -------------------------------------------------------------------------
# Se apunta a la carpeta RP y PKLDataset empareja internamente con AZ (modality="fusion")
train_path_hc = str(DATA_DIR / "HC" / source_thickness / "RP")

sample_sizes = [20, 50, 100, 200, 300, 400, 500]
subset_names = [f"train_{size}" for size in sample_sizes]

# Rutas de evaluación: los 4 espesores restantes de HC (100% de muestras)
val_paths = [
    str(DATA_DIR / "HC" / "T185" / "RP"),
    str(DATA_DIR / "HC" / "T188" / "RP"),
    str(DATA_DIR / "HC" / "T191" / "RP"),
    str(DATA_DIR / "HC" / "T194" / "RP"),
]

# Cargar dataset completo de HC T197 en modo fusion
full_hc_dataset = PKLDataset(folder_path=train_path_hc, modality="fusion")

# Contenedor de resultados: results[subset_name][val_path] -> lista de precisiones
results = {s_name: {vp: [] for vp in val_paths} for s_name in subset_names}

# -------------------------------------------------------------------------
# Verificación de rutas
# -------------------------------------------------------------------------
print("Verifying folder paths for Late Fusion (HC -> HC)...")
for p in [train_path_hc] + val_paths:
    exists = os.path.exists(p)
    print(f"Path ({p}): {'FOUND' if exists else 'MISSING'}")

print(f"\nHC Source Training Path (Fusion): {train_path_hc}")
print(f"HC Total Available Samples: {len(full_hc_dataset)}")
print(f"HC Training Subsets to Evaluate: {subset_names}")
print(f"HC Target Evaluation Thicknesses: {[Path(vp).parent.name for vp in val_paths]}")
print(f"Total Random Seeds: {len(seeds)}")

Verifying folder paths for Late Fusion (HC -> HC)...
Path (c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\HC\T197\RP): FOUND
Path (c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\HC\T185\RP): FOUND
Path (c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\HC\T188\RP): FOUND
Path (c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\HC\T191\RP): FOUND
Path (c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\HC\T194\RP): FOUND

HC Source Training Path (Fusion): c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\HC\T197\RP
HC Total Available Samples: 5010
HC Training Subsets to Evaluate: ['train_20', 'train_50', 'train_100', 'train_200', 'trai

## 2. Configuration (Fusion)

In [2]:
import sys
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

# -------------------------------------------------------------------------
# Path resolution
# -------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent if "__file__" not in globals() else Path(__file__).resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pkldataset import PKLDataset, class_to_onehot

# -------------------------------------------------------------------------
# Stratified Sampling Helper for HC Subsets
# -------------------------------------------------------------------------
def get_balanced_transfer_subset(dataset, total_samples, seed=42):
    """
    Extracts an exactly class-balanced subset from a PKLDataset or Subset instance.
    For example: 20 samples -> 2 samples per wear class (R05..R50).
    Ensures indices are native integers to avoid collation issues in DataLoader.
    """
    np.random.seed(seed)
    num_classes = 10
    samples_per_class = total_samples // num_classes
    
    if isinstance(dataset, Subset):
        base_dataset = dataset.dataset
        available_indices = list(dataset.indices)
    else:
        base_dataset = dataset
        available_indices = list(range(len(dataset.files)))
    
    # Map subset indices by wear class (R05 to R50)
    class_indices = {i: [] for i in range(num_classes)}
    for idx in available_indices:
        filename = base_dataset.files[int(idx)]
        class_label = filename.split('_')[0]
        c_idx = int(np.argmax(class_to_onehot[class_label]))
        class_indices[c_idx].append(int(idx))
        
    selected_indices = []
    for c_idx, indices in class_indices.items():
        if len(indices) < samples_per_class:
            raise ValueError(f"No hay suficientes muestras en la clase {c_idx}: {len(indices)} disponibles vs {samples_per_class} pedidas.")
        chosen = np.random.choice(indices, size=samples_per_class, replace=False)
        selected_indices.extend([int(x) for x in chosen])
        
    return Subset(base_dataset, selected_indices)

# -------------------------------------------------------------------------
# Dataset Paths & Configurations (Late Fusion: HC T197 -> Other HC Targets)
# -------------------------------------------------------------------------
# Source domain training base path: HC T197 (pairs RP and AZ automatically)
source_thickness = "T197"
train_path_hc = str(DATA_DIR / "HC" / source_thickness / "RP")

# Sample sizes extracted from HC (T197) to train models from scratch
sample_sizes = [20, 50, 100, 200, 300, 400, 500]
subset_names = [f"train_{size}" for size in sample_sizes]

# Evaluation paths: independent HC sheet thicknesses (100% of samples)
val_paths = [
    str(DATA_DIR / "HC" / "T185" / "RP"),
    str(DATA_DIR / "HC" / "T188" / "RP"),
    str(DATA_DIR / "HC" / "T191" / "RP"),
    str(DATA_DIR / "HC" / "T194" / "RP"),
]

# Random seeds for reproducibility across runs (10 seeds)
seeds = [101, 202, 303, 404, 505, 606, 707, 808, 909, 1001]

# Load full HC source dataset in fusion mode
full_hc_dataset = PKLDataset(folder_path=train_path_hc, modality="fusion")

# Results container for subset evaluations: results[subset_name][val_path] -> list of accuracies
results = {s_name: {vp: [] for vp in val_paths} for s_name in subset_names}

## 3. Validation Preloading - Fusion

In [3]:
import os
import torch
from torch.utils.data import DataLoader
from src.pkldataset import PKLDataset

# -------------------------------------------------------------------------
# Preload full HC evaluation datasets into RAM (Fusion: 2 channels)
# -------------------------------------------------------------------------
print("Preloading full HC evaluation datasets into RAM (Fusion: 2 channels)...")

# Asignar workers de CPU de forma segura (4 a 8 es ideal para I/O de archivos)
cpu_cores = os.cpu_count() or 4
num_workers = min(8, cpu_cores)
cached_val_tensors = {}

for vp in val_paths:
    thickness_name = vp.split("/")[-2] if "/" in vp else vp.split("\\")[-2]
    print(f"  -> Caching HC_{thickness_name}...", end=" ", flush=True)

    temp_ds = PKLDataset(folder_path=vp, modality="fusion")
    
    # num_workers > 0 paraleliza la apertura y deserialización de los PKL
    temp_loader = DataLoader(
        temp_ds,
        batch_size=512,            # Lote más grande para reducir overhead de colación
        shuffle=False,
        num_workers=num_workers,
        pin_memory=False,          # Mantener en False durante el caching en RAM principal
        persistent_workers=False
    )
    
    all_inputs, all_targets = [], []
    for x, y in temp_loader:
        all_inputs.append(x)
        all_targets.append(y)
        
    cached_val_tensors[vp] = {
        "inputs": torch.cat(all_inputs, dim=0),
        "targets": torch.cat(all_targets, dim=0)
    }
    
    n_samples = cached_val_tensors[vp]["inputs"].shape[0]
    print(f"Listo ({n_samples} muestras cargadas).")

print("\nHC evaluation datasets successfully preloaded into RAM.")
print(f"HC Source Training Path (Fusion): {train_path_hc}")
print(f"HC Total Available Samples: {len(full_hc_dataset)}")
print(f"HC Training Subsets to Evaluate: {subset_names}")
print(f"HC Evaluation Paths (Fusion): {val_paths}")
print(f"Total Random Seeds: {len(seeds)}")

Preloading full HC evaluation datasets into RAM (Fusion: 2 channels)...
  -> Caching HC_T185... Listo (5010 muestras cargadas).
  -> Caching HC_T188... Listo (5010 muestras cargadas).
  -> Caching HC_T191... Listo (5010 muestras cargadas).
  -> Caching HC_T194... Listo (5009 muestras cargadas).

HC evaluation datasets successfully preloaded into RAM.
HC Source Training Path (Fusion): c:\Users\antho\OneDrive\Escritorio\Erstsemester_Anthony\Bachelorthesis\Training_Daten_Stanzen\Data\Data\data\HC\T197\RP
HC Total Available Samples: 5010
HC Training Subsets to Evaluate: ['train_20', 'train_50', 'train_100', 'train_200', 'train_300', 'train_400', 'train_500']
HC Evaluation Paths (Fusion): ['c:\\Users\\antho\\OneDrive\\Escritorio\\Erstsemester_Anthony\\Bachelorthesis\\Training_Daten_Stanzen\\Data\\Data\\data\\HC\\T185\\RP', 'c:\\Users\\antho\\OneDrive\\Escritorio\\Erstsemester_Anthony\\Bachelorthesis\\Training_Daten_Stanzen\\Data\\Data\\data\\HC\\T188\\RP', 'c:\\Users\\antho\\OneDrive\\Escri

## 4. Training and Evaluation - Fusion

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

# -------------------------------------------------------------------------
# Dataset Wrapper para aislar canales (0: RP, 1: AZ) sin duplicar memoria
# -------------------------------------------------------------------------
class ChannelSelectorDataset(Dataset):
    def __init__(self, base_subset, channel_idx):
        self.base_subset = base_subset
        self.channel_idx = channel_idx

    def __len__(self):
        return len(self.base_subset)

    def __getitem__(self, idx):
        x, y = self.base_subset[idx]
        # x shape original: (2, L) -> extrae canal individual: (1, L)
        x_channel = x[self.channel_idx:self.channel_idx + 1, :]
        return x_channel, y

# -------------------------------------------------------------------------
# Fast Evaluation Helper para Decision-Level Late Fusion (Softmax Ensemble)
# -------------------------------------------------------------------------
def fast_eval_decision_fusion(model_rp, model_az, inputs_tensor, targets_tensor, alpha=0.5, batch_size=256, device="cuda"):
    """
    Combina las predicciones calculando las probabilidades Softmax independientes:
    P_final = alpha * P_AZ + (1 - alpha) * P_RP
    """
    model_rp.eval()
    model_az.eval()
    correct = 0
    total = targets_tensor.size(0)

    with torch.no_grad():
        for i in range(0, total, batch_size):
            x = inputs_tensor[i:i + batch_size].to(device)
            y = targets_tensor[i:i + batch_size].to(device)

            x_rp = x[:, 0:1, :]
            x_az = x[:, 1:2, :]

            # Obtención de probabilidades normalizadas
            probs_rp = F.softmax(model_rp(x_rp), dim=1)
            probs_az = F.softmax(model_az(x_az), dim=1)

            # Promediado ponderado de decisiones
            fused_probs = (1.0 - alpha) * probs_rp + alpha * probs_az
            preds = fused_probs.argmax(dim=1)

            targets_idx = y.argmax(dim=1)
            correct += (preds == targets_idx).sum().item()

    return 100.0 * correct / total

# -------------------------------------------------------------------------
# Bucle de Entrenamiento y Evaluación: Decision-Level Late Fusion
# -------------------------------------------------------------------------
# alpha=0.5 otorga el mismo peso a ambos sensores en la decisión final
WEIGHT_AZ = 0.5

for seed in seeds:
    print(f"\n==========================================")
    print(f">>> Running Decision-Level Late Fusion (alpha_AZ={WEIGHT_AZ}) | Seed: {seed}")
    print(f"==========================================")
    set_seed(seed)

    for s_name, n_samples in zip(subset_names, sample_sizes):
        print(f"\n--- Training on HC Subset: {s_name} ({n_samples} samples) ---")

        # 1. Extraer subconjunto balanceado desde HC T197
        base_subset = get_balanced_transfer_subset(
            dataset=full_hc_dataset,
            total_samples=n_samples,
            seed=seed
        )
        batch_sz = min(32, n_samples)

        # 2. Generar vistas independientes de 1 canal para RP y AZ
        ds_rp = ChannelSelectorDataset(base_subset, channel_idx=0)
        ds_az = ChannelSelectorDataset(base_subset, channel_idx=1)

        loader_rp = DataLoader(ds_rp, batch_size=batch_sz, shuffle=True)
        loader_az = DataLoader(ds_az, batch_size=batch_sz, shuffle=True)

        # 3. Entrenar Modelo de Fuerza (RP) - 1 canal
        model_rp = get_model(input_channels=1).to(device)
        opt_rp = optim.Adam(model_rp.parameters(), lr=1e-3, weight_decay=1e-5)
        sched_rp = optim.lr_scheduler.StepLR(opt_rp, step_size=25, gamma=0.1)
        model_rp = train_model(
            model=model_rp,
            loader=loader_rp,
            criterion=criterion,
            optimizer=opt_rp,
            scheduler=sched_rp,
            num_epochs=100,
            device=device
        )

        # 4. Entrenar Modelo de Aceleración (AZ) - 1 canal
        model_az = get_model(input_channels=1).to(device)
        opt_az = optim.Adam(model_az.parameters(), lr=1e-3, weight_decay=1e-5)
        sched_az = optim.lr_scheduler.StepLR(opt_az, step_size=25, gamma=0.1)
        model_az = train_model(
            model=model_az,
            loader=loader_az,
            criterion=criterion,
            optimizer=opt_az,
            scheduler=sched_az,
            num_epochs=100,
            device=device
        )

        # 5. Evaluación por fusión de decisiones en los espesores objetivo
        for vp in val_paths:
            acc = fast_eval_decision_fusion(
                model_rp=model_rp,
                model_az=model_az,
                inputs_tensor=cached_val_tensors[vp]["inputs"],
                targets_tensor=cached_val_tensors[vp]["targets"],
                alpha=WEIGHT_AZ,
                device=device
            )
            results[s_name][vp].append(acc)
            thickness_name = vp.split("/")[-2] if "/" in vp else vp.split("\\")[-2]
            print(f"[Seed {seed}] {s_name} -> HC_{thickness_name} (Decision Fusion): {acc:.2f}%")


>>> Running Decision-Level Late Fusion (alpha_AZ=0.5) | Seed: 101

--- Training on HC Subset: train_20 (20 samples) ---
Epoch 1/100, Loss: 2.3542
Epoch 2/100, Loss: 3.7490
Epoch 3/100, Loss: 2.8780
Epoch 4/100, Loss: 2.1987
Epoch 5/100, Loss: 1.9472
Epoch 6/100, Loss: 2.3375
Epoch 7/100, Loss: 1.6389
Epoch 8/100, Loss: 1.3614
Epoch 9/100, Loss: 1.2405
Epoch 10/100, Loss: 1.4770
Epoch 11/100, Loss: 1.3615
Epoch 12/100, Loss: 1.0528
Epoch 13/100, Loss: 0.6930
Epoch 14/100, Loss: 0.8065
Epoch 15/100, Loss: 1.0970
Epoch 16/100, Loss: 0.8339
Epoch 17/100, Loss: 0.7139
Epoch 18/100, Loss: 1.1053
Epoch 19/100, Loss: 0.9626
Epoch 20/100, Loss: 1.3254
Epoch 21/100, Loss: 1.0385
Epoch 22/100, Loss: 0.7453
Epoch 23/100, Loss: 0.6898
Epoch 24/100, Loss: 1.0051
Epoch 25/100, Loss: 0.8378
Epoch 26/100, Loss: 0.5406
Epoch 27/100, Loss: 0.6978
Epoch 28/100, Loss: 0.5988
Epoch 29/100, Loss: 0.5160
Epoch 30/100, Loss: 0.7561
Epoch 31/100, Loss: 0.4599
Epoch 32/100, Loss: 0.6480
Epoch 33/100, Loss: 0.91

## 5. Summary of Results - Fusion

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

# -------------------------------------------------------------------------
# Compile evaluation summary across evaluated HC sheet thicknesses (Decision Fusion)
# -------------------------------------------------------------------------
summary_data = []

for s_name in subset_names:
    row = {"Subset": s_name}
    for vp in val_paths:
        accs = results[s_name][vp]
        mean_acc = np.mean(accs)
        std_acc = np.std(accs)

        # Extracción limpia y multiplataforma del nombre del espesor (e.g., 'T185')
        col_name = Path(vp).parent.name
        row[col_name] = f"{mean_acc:.2f} ± {std_acc:.2f}"
    summary_data.append(row)

# Orden de columnas para HC -> HC (evaluación cross-thickness)
column_order = ["Subset", "T185", "T188", "T191", "T194"]
df_decision_fusion = pd.DataFrame(summary_data)[column_order]

# Exportar resultados consolidados a CSV identificando 'DecisionFusion'
output_csv = "wear_DecisionFusion_HC_train_HC_eval_summary.csv"
df_decision_fusion.to_csv(output_csv, index=False)

print("Consolidated Decision-Level Late Fusion (RP + AZ) results (HC Train -> HC Evaluation):")
display(df_decision_fusion)
print(f"Results successfully saved to: {output_csv}")

Consolidated Decision-Level Late Fusion (RP + AZ) results (HC Train -> HC Evaluation):


,Subset,T185,T188,T191,T194
0,train_20,74.14 ± 8.61,76.48 ± 12.93,80.86 ± 11.99,81.01 ± 11.86
1,train_50,80.10 ± 2.78,88.99 ± 4.11,91.89 ± 2.92,90.70 ± 2.78
2,train_100,81.21 ± 3.53,88.30 ± 3.71,91.27 ± 3.04,90.73 ± 2.58
3,train_200,78.62 ± 3.52,88.73 ± 3.38,90.95 ± 2.46,89.89 ± 1.77
4,train_300,81.64 ± 3.18,92.25 ± 2.85,92.35 ± 1.95,89.78 ± 1.65
5,train_400,81.82 ± 2.88,92.01 ± 3.74,91.74 ± 1.93,89.93 ± 2.05
6,train_500,80.63 ± 1.95,91.64 ± 2.45,91.98 ± 1.58,89.30 ± 1.47


Results successfully saved to: wear_DecisionFusion_HC_train_HC_eval_summary.csv
